# Lab 3 · Planning and reasoning

This lab has two halves. The first compares **classical planning** (search over states) with having an LLM plan. The second applies three agent planning patterns to real work: reviewing 3 contracts against a company policy of 8 rules, measuring which pattern is most accurate and what it costs.

| Part | What you do | Time |
|---|---|---|
| 1 | Classical planning with breadth-first search | 10 min |
| 2 | The LLM as planner, and a plan checker | 10 min |
| 3 | The contract task, and the single-call approach | 5 min |
| 4 | Reasoning before deciding | 8 min |
| 5 | Plan-and-Execute with LangGraph | 15 min |
| 6 | Reflection: draft, then review | 10 min |
| 7 | Comparison | 5 min |
| 8 | Stretch: ReAct and self-consistency | If time allows |

Parts 3–6 call the LLM dozens of times. The times noted in the cells come from the test machine; yours may be slower.

Look for `# TODO` in the code. Before you start, save this file as `lab3_planning.mine.ipynb`.

## 1. Classical planning with breadth-first search (10 min)

In the style of AIMA chapter 10, a planning problem has **states** (the set of facts that are true), **actions** with preconditions (`pre`) and effects that add (`add`) or delete (`del`) facts, a start state, and a **goal**. A planner searches for a sequence of actions that leads from the start to the goal.

The domain below is an invoice approval process. Note that `submit_for_approval` **deletes** `editable`: once submitted, the amount can no longer be corrected.

**Your job:** write `applicable` (an action can run when all of its `pre` facts are true in the state) and `apply` (remove the `del` facts, then add the `add` facts).

In [ ]:
import time
from collections import deque

ACTIONS = {
    "extract_fields":      {"pre": {"have_text"},                   "add": {"have_fields"},    "del": set()},
    "check_total":         {"pre": {"have_fields"},                 "add": {"total_checked"},  "del": set()},
    "correct_amount":      {"pre": {"have_fields", "editable"},     "add": {"amount_correct"}, "del": set()},
    "submit_for_approval": {"pre": {"total_checked"},               "add": {"submitted"},      "del": {"editable"}},
    "reopen":              {"pre": {"submitted"},                   "add": {"editable"},       "del": {"submitted"}},
    "approve":             {"pre": {"submitted", "amount_correct"}, "add": {"approved"},       "del": set()},
}
START, GOAL = {"have_text", "editable"}, {"approved"}


def applicable(action, state, actions=ACTIONS):
    # TODO: return True if every precondition of the action is in the state
    return True


def apply(action, state, actions=ACTIONS):
    # TODO: return a new frozenset: the state minus the action's "del" facts, plus its "add" facts
    return frozenset(state)


def bfs_plan(start, goal, actions=ACTIONS):
    """Breadth-first search over states. Returns (shortest plan, number of states explored)."""
    start = frozenset(start)
    frontier, seen = deque([(start, [])]), {start}
    while frontier:
        state, plan = frontier.popleft()
        if goal <= state:
            return plan, len(seen)
        for action in actions:
            if applicable(action, state, actions):
                nxt = apply(action, state, actions)
                if nxt not in seen:
                    seen.add(nxt)
                    frontier.append((nxt, plan + [action]))
    return None, len(seen)


def validate(plan, start, goal, actions=ACTIONS):
    """Check a plan step by step. Returns 'valid' or a message saying where it fails."""
    state = frozenset(start)
    for i, action in enumerate(plan, 1):
        if action not in actions:
            return f"step {i}: there is no action called {action}"
        if not applicable(action, state, actions):
            return f"step {i} ({action}) cannot run: it needs {sorted(actions[action]['pre'] - state)}, which is not true at that point"
        state = apply(action, state, actions)
    return "valid" if goal <= state else f"the goal is not reached: {sorted(goal - state)} is still missing"


plan, explored = bfs_plan(START, GOAL)
if plan is None:
    print(f"no plan found after exploring {explored} states (have you written applicable and apply?)")
else:
    print("plan:", plan, f"\nexplored {explored} states\ncheck:", validate(plan, START, GOAL))
print("a wrong order:", validate(["extract_fields", "check_total", "submit_for_approval", "correct_amount", "approve"], START, GOAL))

## 2. The LLM as planner, and a plan checker (10 min)

Now the LLM plans instead of search, from a text description of the domain. We check its plan with `validate`; if it is wrong, we send back where it failed and ask for a fix, up to 4 rounds. This pattern pairs an LLM with an external checker (sometimes called LLM-Modulo).

In [ ]:
from typing import Literal

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from pydantic import BaseModel, Field

from agentlab import get_llm, release_model

llm = get_llm()


def describe(actions):
    lines = []
    for name, a in actions.items():
        line = f"- {name}: needs {sorted(a['pre'])}; makes true {sorted(a['add'])}"
        if a["del"]:
            line += f"; makes false {sorted(a['del'])}"
        lines.append(line)
    return "\n".join(lines)


def llm_plan(start, goal, actions=ACTIONS, rounds=4):
    """Ask the LLM for a plan, check it, and send back the error until it is valid or rounds run out."""
    class Plan(BaseModel):
        steps: list[Literal[tuple(actions)]] = Field(description="The actions in the order to perform them")

    messages = [
        SystemMessage("You plan a sequence of actions. An action can run only when everything it needs is true at that moment. "
                      "Its results become true, and anything it makes false stops being true."),
        HumanMessage(f"Actions:\n{describe(actions)}\n\nTrue now: {sorted(start)}\nGoal: {sorted(goal)}\n\nGive the shortest plan."),
    ]
    for attempt in range(1, rounds + 1):
        plan = llm.with_structured_output(Plan).invoke(messages)
        verdict = validate(plan.steps, start, goal, actions)
        print(f"  round {attempt}: {plan.steps} -> {verdict}")
        if verdict == "valid":
            return plan.steps
        messages += [AIMessage(str(plan.steps)), HumanMessage(f"That plan is invalid: {verdict}. Give a corrected plan.")]
    return None


print("LLM planner:")
llm_plan(START, GOAL)

The LLM probably plans correctly in the first round. But is it reasoning from the conditions, or is it just familiar with names like `submit_for_approval` and `approve`?

The cell below changes the **names** of every action and fact to meaningless words. The structure (preconditions and effects) is exactly the same. Then both search and the LLM plan again (about 30 seconds).

In [ ]:
NONSENSE = {"extract_fields": "plim", "check_total": "dorve", "correct_amount": "sneb", "submit_for_approval": "quarn",
            "reopen": "tobble", "approve": "wexit", "have_text": "zorg", "have_fields": "blick", "total_checked": "frap",
            "editable": "glent", "amount_correct": "mibber", "submitted": "harl", "approved": "yuxo"}
rename = lambda names: {NONSENSE[n] for n in names}
MYSTERY = {NONSENSE[name]: {part: rename(facts) for part, facts in a.items()} for name, a in ACTIONS.items()}
M_START, M_GOAL = rename(START), rename(GOAL)

plan, explored = bfs_plan(M_START, M_GOAL, MYSTERY)
print("search:", plan, "->", validate(plan, M_START, M_GOAL, MYSTERY) if plan else "no plan found")
print("LLM planner:")
llm_plan(M_START, M_GOAL, MYSTERY)
release_model()

On the test machine the LLM failed all 4 rounds on the nonsense domain, even with an error message every round, while search found the same plan immediately. Your result may differ a little. The point: **an LLM plans from familiarity with language**, while a classical planner plans from the rules and guarantees a correct plan under the domain model. A good system uses both: the LLM proposes and code checks.

## 3. The contract task, and the single-call approach (5 min)

The company policy has 8 rules. Some apply to every contract; others only to service contracts, purchases of goods, contracts over 500,000 THB, or suppliers that process personal data. We review 3 contracts (`CLASS_SET`) and measure precision (how much of what is reported is right) and recall (how much of what is really wrong is found).

The first approach sends the whole policy and the contract together and asks once (about 15 seconds).

In [ ]:
import pandas as pd

from agentlab.contracts import CLASS_SET, RULES, evaluate, load_contracts, load_labels, policy_text, score_line

contracts, labels = load_contracts(), load_labels()
print(policy_text())
print("\n--- c1 ---\n" + contracts["c1"])

RuleId = Literal["R1", "R2", "R3", "R4", "R5", "R6", "R7", "R8"]
REVIEWER = "You review contracts against the company contract policy. A rule counts only if it applies to this kind of contract."


class Violations(BaseModel):
    violated: list[RuleId] = Field(description="IDs of the policy rules this contract breaks; empty if it breaks none")


results, calls = {}, {"n": 0}


def ask(schema, system, user, model=None):
    """One structured LLM call, counted. Retried once: a small model sometimes fails to finish the JSON."""
    chain = (model or llm).with_structured_output(schema)
    messages = [SystemMessage(system), HumanMessage(user)]
    calls["n"] += 1
    try:
        return chain.invoke(messages)
    except Exception:
        calls["n"] += 1
        return chain.invoke(messages)


def run_pattern(name, review):
    """Run review(contract_id) -> set of rule IDs on CLASS_SET, then score it."""
    calls["n"], start = 0, time.time()
    found = {cid: review(cid) for cid in CLASS_SET}
    release_model()
    table = evaluate(found, CLASS_SET)
    results[name] = {**score_line(name, table), "llm_calls": calls["n"], "seconds": round(time.time() - start)}
    print(results[name])
    return table


check_llm = get_llm(num_predict=300)  # a short cap: checking one rule needs a sentence, not a page


def check_rule(cid, rid, schema, system="You check one contract against one policy rule."):
    """Ask whether one contract satisfies one rule; schema decides what the answer looks like (parts 4 and 5)."""
    return ask(schema, system, f"Rule {rid}: {RULES[rid]['text']}\n\nContract:\n{contracts[cid]}\n\nDoes the contract satisfy this rule?", model=check_llm)


def single_call(cid):
    out = ask(Violations, REVIEWER, f"Policy:\n{policy_text()}\n\nContract:\n{contracts[cid]}\n\nWhich rules does the contract break?")
    return set(out.violated)


run_pattern("single call", single_call)

## 4. Reasoning before deciding (8 min)

In structured output the model writes the fields in schema order. If the answer field (`complies`) comes first, the model must decide before it has thought. If a reasoning field comes first, it gets to "think aloud" before deciding: chain-of-thought, in the shape of a schema.

The cell below checks 4 hard cases with both schemas and counts which gets more right (about 1 minute).

In [ ]:
class AnswerFirst(BaseModel):
    complies: bool = Field(description="true if the contract satisfies the rule")
    reasoning: str = Field(description="One short sentence explaining why")


class ReasonFirst(BaseModel):
    reasoning: str = Field(description="One short sentence: what the contract says on this point, and whether that meets the rule")
    complies: bool = Field(description="true if the contract satisfies the rule")


hard_cases = [("c5", "R2"), ("c5", "R7"), ("c3", "R7"), ("c1", "R2")]
rows = []
for cid, rid in hard_cases:
    truth = rid not in labels.loc[cid, "violated_rules"]
    for schema in (AnswerFirst, ReasonFirst):
        out = check_rule(cid, rid, schema)
        rows.append({"case": f"{cid}/{rid}", "schema": schema.__name__, "correct": out.complies == truth, "reasoning": out.reasoning[:90]})
release_model()
table = pd.DataFrame(rows)
print(table.groupby("schema")["correct"].sum().to_string())
table

Read the `reasoning` column of the wrong rows. Sometimes the model writes correct reasoning but gives the opposite answer, because it had already decided. On the test machine the answer-first schema got 1 of 4 right and the reason-first schema got 4 of 4. Your result may differ, since 4 cases is a small sample.

**Thinking (optional, about 1.5 minutes):** the course model can "think" before answering if that mode is switched on (Modules 0 and 1). The cell below gives the answer-first schema another try at c5/R2 with thinking on. Watch the time and the token count.

In [ ]:
thinking_llm = get_llm(reasoning=True, num_predict=4096)  # thinking needs room: it can use a thousand tokens
start = time.time()
raw = thinking_llm.with_structured_output(AnswerFirst, include_raw=True).invoke(
    [SystemMessage("You check one contract against one policy rule."),
     HumanMessage(f"Rule R2: {RULES['R2']['text']}\n\nContract:\n{contracts['c5']}\n\nDoes the contract satisfy this rule?")])
release_model()
print(f"answer: {raw['parsed']}\n{time.time() - start:.0f}s, {raw['raw'].usage_metadata['output_tokens']} output tokens")

## 5. Plan-and-Execute with LangGraph (15 min)

Instead of checking every rule at once, we first plan which rules to check, then check them one at a time:

1. **The LLM extracts facts** from the contract: is it a purchase of goods, is it a service, its value, and whether personal data is involved
2. **Code plans** from those facts: it picks only the rules that apply to this contract
3. **The LLM checks one rule at a time**, with the reason-first schema

**Your job (1):** fill in the conditions in `plan_rules` for R3, R4, R5 and R8, following each rule's scope in the policy text above.

In [ ]:
class Facts(BaseModel):
    # Two yes/no questions, not one "service / goods / other" choice: in testing the small model
    # labelled a computer purchase "service" every time with the choice, and got it right with yes/no.
    buys_goods: bool = Field(description="true if the company is buying products such as computers, equipment or materials")
    is_service: bool = Field(description="true if the supplier performs ongoing work such as maintenance, cleaning, consulting or hosting")
    total_value_thb: float = Field(description="Total value of the contract in THB for its whole term")
    processes_personal_data: bool = Field(description="true if the supplier will handle personal data, such as customers' names or phone numbers")


def plan_rules(facts: Facts) -> list[str]:
    """The plan: which policy rules apply to a contract with these facts."""
    rules = ["R1", "R2", "R6", "R7"]  # these apply to every contract
    # TODO: add R3 if the total value is over 500,000 THB
    # TODO: add R4 if it is a service contract (facts.is_service)
    # TODO: add R5 if the supplier processes personal data
    # TODO: add R8 if it is a contract to buy goods (facts.buys_goods)
    return sorted(rules)


for cid in CLASS_SET:
    facts = ask(Facts, "You extract facts from a contract.", contracts[cid])
    print(cid, facts, "->", plan_rules(facts), "| answer key:", sorted(labels.loc[cid, "applicable_rules"]))
release_model()

Why does code plan, instead of letting the LLM choose the rules? In testing, when the LLM chose the applicable rules from the policy text, it included rules that do not apply, such as the goods warranty rule for a service contract, producing many false reports. Extracting three facts is something the LLM does accurately; comparing them with conditions is a job for code, as in Modules 1–2.

### LangGraph
Now we assemble the steps as a graph in LangGraph:

- **State** is a dict every node reads and updates (`ReviewState`)
- A **node** is a function that takes the state and returns the parts that change
- An **edge** says which node runs after which, and a **conditional edge** chooses the next node from the state

**Your job (2):** write `route` so it goes to `check_next_rule` while there are rules left in the plan, and to `report` once the plan is empty. Then run the cell (about 3 minutes).

In [ ]:
from typing import TypedDict

from langgraph.graph import END, START, StateGraph


class ReviewState(TypedDict, total=False):
    contract_id: str
    facts: Facts
    plan: list[str]          # rules still to check
    checks: dict[str, bool]  # rule -> complies
    violations: list[str]


def extract_facts(state):
    return {"facts": ask(Facts, "You extract facts from a contract.", contracts[state["contract_id"]])}


def make_plan(state):
    return {"plan": plan_rules(state["facts"]), "checks": {}}


class RuleCheck(BaseModel):  # reason first, kept brief: Plan-and-Execute makes many of these calls
    reasoning: str = Field(description="At most 15 words: the relevant clause, and whether it meets the rule")
    complies: bool = Field(description="true if the contract satisfies the rule")


def check_next_rule(state):
    rid, rest = state["plan"][0], state["plan"][1:]
    result = check_rule(state["contract_id"], rid, RuleCheck, "You check one contract against one policy rule. Keep the reasoning brief.")
    return {"plan": rest, "checks": {**state["checks"], rid: result.complies}}


def report(state):
    return {"violations": sorted(rid for rid, ok in state["checks"].items() if not ok)}


def route(state):
    # TODO: return "check_next_rule" if state["plan"] still has rules, otherwise "report"
    return "report"


graph = StateGraph(ReviewState)
graph.add_node("extract_facts", extract_facts)
graph.add_node("make_plan", make_plan)
graph.add_node("check_next_rule", check_next_rule)
graph.add_node("report", report)
graph.add_edge(START, "extract_facts")
graph.add_edge("extract_facts", "make_plan")
graph.add_conditional_edges("make_plan", route, ["check_next_rule", "report"])  # the list names the possible next nodes
graph.add_conditional_edges("check_next_rule", route, ["check_next_rule", "report"])
graph.add_edge("report", END)
reviewer = graph.compile()
print(reviewer.get_graph().draw_mermaid())


def plan_and_execute(cid):
    final = reviewer.invoke({"contract_id": cid}, {"recursion_limit": 30})
    return set(final["violations"])


run_pattern("plan-and-execute", plan_and_execute)

## 6. Reflection: draft, then review (10 min)

Reflection uses two calls. A **drafter** lists the rules the contract breaks with a short reason, then a **reviewer** reads that draft alongside the contract and the policy, removes what is wrong and adds what was missed. This is the evaluator-optimizer pattern from Module 1.

**Your job:** write the reviewer's instruction in `REVIEW_INSTRUCTION`. Make it clear when a finding should be kept, when it should be removed, and that missed rules should be added. Then run the cell (about 1.5 minutes).

In [ ]:
class Finding(BaseModel):
    rule_id: RuleId
    reason: str = Field(description="At most 15 words")


class Draft(BaseModel):
    findings: list[Finding] = Field(description="Every rule the contract breaks; empty if none")


# TODO: write the reviewer's instruction
REVIEW_INSTRUCTION = "Review the findings."


def reflection(cid):
    draft = ask(Draft, REVIEWER, f"Policy:\n{policy_text()}\n\nContract:\n{contracts[cid]}\n\nWhich rules does the contract break, and why?")
    listed = "\n".join(f"- {f.rule_id}: {f.reason}" for f in draft.findings) or "(none)"
    review = ask(Violations, REVIEW_INSTRUCTION,
                 f"Policy:\n{policy_text()}\n\nContract:\n{contracts[cid]}\n\nFirst reviewer's findings:\n{listed}\n\nWhich rules does the contract really break?")
    print(f"  {cid}: draft {sorted(f.rule_id for f in draft.findings)} -> reviewed {sorted(review.violated)}")
    return set(review.violated)


run_pattern("reflection", reflection)

## 7. Comparison (5 min)

The table below gathers the results of every pattern run so far. F1 combines precision and recall into one number. Read it alongside the number of LLM calls and the time, which are the cost.

In [ ]:
pd.DataFrame(results.values()).set_index("pattern")[["precision", "recall", "f1", "fp", "fn", "llm_calls", "seconds"]]

## 8. Stretch

### 8.1 ReAct (about 3–5 minutes)
ReAct from Lab 2: an agent with tools reads the contract and the policy and decides for itself what to check, with no plan in advance. In testing on 5 contracts it often stopped before checking everything: high precision, low recall.

### 8.2 Self-consistency
Ask the single-call question 3 times at temperature 0.8 and keep only the rules reported in at least 2 of the 3 answers. How does the result change compared with asking once?

In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import tool

from agentlab.contracts import TOOLS as CONTRACT_TOOLS


def react(cid):
    reported = {}

    @tool
    def report_violations(violated: list[RuleId]) -> str:
        """Report the final list of rule IDs this contract breaks (an empty list if none). Call it once, at the end."""
        reported["rules"] = set(violated)
        return "reported"

    agent = create_agent(llm, CONTRACT_TOOLS + [report_violations],
                         system_prompt=REVIEWER + " Read the contract, list the policy rules, check each rule that applies, then call report_violations once.")
    out = agent.invoke({"messages": [HumanMessage(f"Review contract {cid}.")]}, {"recursion_limit": 25})
    calls["n"] += sum(m.type == "ai" for m in out["messages"])  # every AI message is one model call
    return reported.get("rules", set())


run_pattern("ReAct", react)

In [ ]:
voting_llm = get_llm(temperature=0.8)


def self_consistency(cid, samples=3):
    votes = {}
    for _ in range(samples):
        out = ask(Violations, REVIEWER, f"Policy:\n{policy_text()}\n\nContract:\n{contracts[cid]}\n\nWhich rules does the contract break?", model=voting_llm)
        for rid in set(out.violated):
            votes[rid] = votes.get(rid, 0) + 1
    return {rid for rid, n in votes.items() if n >= 2}


run_pattern("self-consistency (3 votes)", self_consistency)
pd.DataFrame(results.values()).set_index("pattern")[["precision", "recall", "f1", "fp", "fn", "llm_calls", "seconds"]]

## Wrap-up and what to hand in

Answer these 4 questions in 2–3 sentences each, and hand them in with a screenshot of the table from part 7.

1. In part 2, why did the LLM plan correctly with meaningful names but fail with meaningless ones, when the structure was the same? What does that tell you about having an LLM plan in real work?
2. In part 4, why can the order of fields in a schema change how accurate the answers are?
3. From the table in part 7, which pattern would you choose to review 200 contracts a day, and which if letting a non-compliant contract through were very expensive?
4. In part 5's Plan-and-Execute, what does the LLM do and what does code do? Why divide it that way?

**From notebook to program:** `contract_review.py` in the same folder runs these patterns on all 5 contracts and records the reasoning behind every rule check. `README.md` shows how to run it and `contract_review.en.md` explains it part by part.

**Project milestone 3:** sketch your group's agent workflow as a graph (nodes and edges). Mark which steps the LLM does and which code does, and where a check or reflection step belongs.